# Classification and Captioning Aircraft Damage Using Pretrained Models (VGG16)


## **Introduction**

In this project, you will classify aircraft damage using a pre-trained VGG16 model and generate captions using a Transformer-based pretrained model.

## **Project Overview**

Aircraft damage detection is essential for maintaining the safety and longevity of aircraft. Traditional manual inspection methods are time-consuming and prone to human error. This project aims to automate the classification of aircraft damage into two categories: "dent" and "crack." For this, we will utilize feature extraction with a pre-trained VGG16 model to classify the damage from aircraft images. Additionally, we will use a pre-trained Transformer model to generate captions and summaries for the images.

## **Aim of the Project**

The goal of this project is to develop an automated model that accurately classifies aircraft damage from images. By the end of the project, you will have trained and evaluated a model that utilizes feature extraction from VGG16 for damage classification. This model will be applicable in real-world damage detection within the aviation industry. Furthermore, the project will showcase how we can use a Transformer-based model to caption and summarize images, providing a detailed description of the damage.

## **Final Output**

- A trained model capable of classifying aircraft images into "dent" and "crack" categories, enabling automated aircraft damage detection.
- A Transformer-based model that generates captions and summaries of images


## Objectives

After you complete the project, you will be able to:

- Use the VGG16 model for image classification.
- Prepare and preprocess image data for a machine learning task.
- Evaluate the model’s performance using appropriate metrics.
- Visualize model predictions on test data.
- Use a custom Keras layer. 


 ### Task List
To achieve the above objectives, you will complete the following tasks:

- Task 1: Create a `valid_generator` using the `valid_datagen` object
- Task 2: Create a `test_generator` using the `test_datagen` object
- Task 3: Load the VGG16 model
- Task 4: Compile the model
- Task 5: Train the model
- Task 6: Plot accuracy curves for training and validation sets 
- Task 7: Visualizing the results 
- Task 8: Implement a Helper Function to Use the Custom Keras Layer
- Task 9: Generate a caption for an image using the using BLIP pretrained model
- Task 10: Generate a summary of an image using BLIP pretrained model

**Note**:.<br>
1. For each task, copy and save the code or output as mentioned in the task for final grading.<br>
2. Download the file after completion of the final project.The file should have both code and output.This will be used for final grading.


In [1]:
import keras
import zipfile
import tensorflow as tf
from keras.layers import Dense, Dropout, Flatten
from keras.applications import VGG16
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential,Model
from keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import random



In [2]:
# Set seed for reproducibility
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

#### Pretrained Model

A pretrained model refers to a machine learning model that has already been trained on a large dataset, typically for a specific task, and is ready for use or fine-tuning on a new task or dataset. The key idea behind a pretrained model is that it has already learned useful patterns or features from the data it was trained on, so you don’t need to start from scratch.

- **ResNet, VGG (Image Classification):** These are pretrained models commonly used for image classification tasks. They have learned from millions of images and can be fine-tuned for specific image-related tasks.

- **BLIP (Image Captioning and Summarization):** BLIP is a pretrained model that can generate captions and summaries for images. It has already been trained on image-text pairs, so it can easily generate descriptive captions for new images.


### Part 1 - Classification Problem: Classifying the defect on the aircraft as 'dent' or 'crack'

The first step is to load and prepare the dataset of aircraft images. These images are labeled either as 'dent' or 'crack'. We will also split the dataset into training, validation, and test sets.

Your goal is to train an algorithm on these images and to predict the labels for images in your test set.

In [3]:
# set parameters
batch_size = 32
epochs = 10
img_rows, img_cols = 224,224
input_shape = (img_rows,img_cols,3)

Extract the Dataset:
Unzip the dataset to the current directory, creating directories for training, testing, and validation splits.


In [4]:
import tarfile
import urllib.request
import os
import shutil

In [5]:
file_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZjXM4RKxlBK9__ZjHBLl5A/aircraft-damage-dataset-v1.tar"
tarfile_name = "aircraft_damage_dataset_v1.tar"
extracted_folder = "aircraft_damage_dataset_v1"

#Download tarfile from url
urllib.request.urlretrieve(file_url, tarfile_name)
print(f"Downloaded {tarfile}, Extracted will Begin Now")

Downloaded <module 'tarfile' from 'c:\\Program Files\\Python311\\Lib\\tarfile.py'>, Extracted will Begin Now


In [6]:
if os.path.exists(extracted_folder):
    print(f"Floder {extracted_folder} already exists. Removing the existing folder")
    shutil.rmtree(extracted_folder)
    print(f"{extracted_folder} removed successfull.")

# extract the downloaded tar file
with tarfile.open(tarfile_name, "r") as tar_ref:
    tar_ref.extractall()
    print(f"{tarfile_name} extracted successfull.")

Floder aircraft_damage_dataset_v1 already exists. Removing the existing folder
aircraft_damage_dataset_v1 removed successfull.
aircraft_damage_dataset_v1.tar extracted successfull.


In [7]:
# Define directories for train, test, and validation splits
train_dir = os.path.join(extracted_folder,'train')
test_dir = os.path.join(extracted_folder,'test')
valid_dir = os.path.join(extracted_folder,"valid")


## 1.2 Data Preprocessing

Create data generators for training, validation, and testing datasets.

First,we will create ImageDataGenerators used for training, validation and testing.
The ImageDataGenerator class is part of Keras. It is a powerful utility for real-time image data augmentation, preprocessing, and feeding data into deep learning models during training. This class is particularly useful when working with image datasets that are too large to fit into memory all at once, or when you want to augment your dataset  to improve model generalization. 

We will create instances of the ImageDataGenerator class. Each instance corresponds to one of the datasets: training, validation, and testing.


In [8]:
train_datagenerator = ImageDataGenerator(rescale=1./255)
test_datagenerator = ImageDataGenerator(rescale=1./255)
valid_datagenerator = ImageDataGenerator(rescale=1./255) 

Next, we use flow_from_directory() method to load the images from directory and generate the training dataset. The flow_from_directory() method is part of the ImageDataGenerator class in Keras, and it plays a crucial role in automating the process of loading, preprocessing, and batching images for training, validation, and testing.
We use the train_datagen object to load and preprocess the training images. Specifically, the flow_from_directory() function is used to read images directly from the directory and generate batches of data that will be fed into the model for training.


In [9]:
# generate training dataset
train_genrator = train_datagenerator.flow_from_directory(
    train_dir,
    batch_size = batch_size,
    target_size = (img_rows,img_cols),
    seed = seed_value,
    class_mode = 'binary',
    shuffle = True
)

Found 300 images belonging to 2 classes.


In [10]:
# validation dataset generator

valid_generator = valid_datagenerator.flow_from_directory(
    valid_dir,
    batch_size = batch_size,
    target_size = (img_rows, img_cols),
    shuffle = False,
    class_mode = 'binary',
    seed = seed_value
)

Found 96 images belonging to 2 classes.


In [11]:
# Test data generator
test_genrator = test_datagenerator.flow_from_directory(
    test_dir,
    seed = seed_value,
    shuffle = False,
    class_mode = 'binary',
    target_size = (img_rows,img_cols),
    batch_size = batch_size
)

Found 50 images belonging to 2 classes.


## 1.3 Model Definition

Here, we define the model architecture by using a pre-trained VGG16 model as the base, adding custom layers on top for binary classification of 'dent' and 'crack' types of damage.


## **Task 3: Load the pre-trained model VGG16**

Set <code>weights='imagenet'</code>,<code>include_top=False</code>,<code>input_shape=(img_rows, img_cols, 3)</code>

Hint: The format should be like:

base_model = VGG16(weights= , include_top= , input_shape=)

****Note: Please copy and save the code of the task as it will be required for submission in the final project. Ensure to submit the response as part of your project submission****


In [12]:
base_model = VGG16(weights= 'imagenet', include_top=False, input_shape=(img_rows,img_cols,3))

In [13]:
print(base_model.layers[-1].output)

<KerasTensor shape=(None, 7, 7, 512), dtype=float32, sparse=False, ragged=False, name=keras_tensor_18>


base-model output shape is *shape=(None, 7, 7, 512)*, but we need a flattern output to connect the output to a fully connected (DENSE) neural network for classification.

In [14]:
# get base_model last layer output
output = base_model.layers[-1].output

# flattern the output suitable for the dense layer
output = keras.layers.Flatten()(output)

# Rebuild the base_model that get input as base_model input and 
# output as flattern output
base_model = Model(base_model.input, output)

# freeze the weights of each layer when train the model
for layer in base_model.layers:
    layer.trainable = False

After using VGG16 as a feature extractor, we add our own classifier on top of the VGG16 model. This involves adding fully connected layers (Dense), activation functions (like ReLU), and sometimes Dropout layers to avoid overfitting.
Here, we are adding two dense layers with 512 units each, followed by a Dropout layer, and finally, a Dense layer with one unit and a sigmoid activation to output the probability for binary classification ("dent" vs "crack").


In [15]:
adam = Adam(learning_rate=0.0001)

In [16]:
# initiate a sequential object as model
model = Sequential()

# configure the model
model.add(base_model)
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))

# compile the model
model.compile(optimizer=adam, loss='binary_crossentropy', metrics=['accuracy'] )

### Model Training

In [22]:
history = model.fit(train_genrator, validation_data=valid_generator, epochs=epochs, verbose=2)

Epoch 1/10
10/10 - 62s - 6s/step - accuracy: 0.9600 - loss: 0.1152 - val_accuracy: 0.7188 - val_loss: 0.4818
Epoch 2/10
10/10 - 64s - 6s/step - accuracy: 0.9767 - loss: 0.0868 - val_accuracy: 0.7292 - val_loss: 0.4712
Epoch 3/10
10/10 - 65s - 6s/step - accuracy: 0.9767 - loss: 0.0807 - val_accuracy: 0.7604 - val_loss: 0.4328
Epoch 4/10
10/10 - 66s - 7s/step - accuracy: 0.9767 - loss: 0.0777 - val_accuracy: 0.7500 - val_loss: 0.5087
Epoch 5/10
10/10 - 67s - 7s/step - accuracy: 0.9833 - loss: 0.0654 - val_accuracy: 0.7500 - val_loss: 0.4303
Epoch 6/10
10/10 - 69s - 7s/step - accuracy: 0.9967 - loss: 0.0504 - val_accuracy: 0.7396 - val_loss: 0.4846
Epoch 7/10
10/10 - 69s - 7s/step - accuracy: 0.9933 - loss: 0.0424 - val_accuracy: 0.7292 - val_loss: 0.5042
Epoch 8/10
10/10 - 69s - 7s/step - accuracy: 0.9967 - loss: 0.0350 - val_accuracy: 0.7188 - val_loss: 0.5018
Epoch 9/10
10/10 - 69s - 7s/step - accuracy: 0.9900 - loss: 0.0409 - val_accuracy: 0.7292 - val_loss: 0.5795
Epoch 10/10
10/10 -

#### Model Evaluation

In [18]:
score = model.evaluate(test_genrator,verbose=2)
score

2/2 - 7s - 4s/step - accuracy: 0.8200 - loss: 0.5777


[0.5776802897453308, 0.8199999928474426]

In [20]:
print("Model Accuracy: {:.3f}%, Model Loss: {:.3f}%".format(100 * score[1],100 * score[0]))

Model Accuracy: 82.000%, Model Loss: 57.768%


In [23]:
training_history = model.history.history
training_history

{'accuracy': [0.9599999785423279,
  0.9766666889190674,
  0.9766666889190674,
  0.9766666889190674,
  0.9833333492279053,
  0.996666669845581,
  0.9933333396911621,
  0.996666669845581,
  0.9900000095367432,
  0.9900000095367432],
 'loss': [0.11524996161460876,
  0.08679495751857758,
  0.08071223646402359,
  0.07769627124071121,
  0.06541630625724792,
  0.0504133477807045,
  0.042381931096315384,
  0.034984029829502106,
  0.04086033254861832,
  0.04151245206594467],
 'val_accuracy': [0.71875,
  0.7291666865348816,
  0.7604166865348816,
  0.75,
  0.75,
  0.7395833134651184,
  0.7291666865348816,
  0.71875,
  0.7291666865348816,
  0.7291666865348816],
 'val_loss': [0.4817758798599243,
  0.47123458981513977,
  0.43277645111083984,
  0.5087171196937561,
  0.4303319752216339,
  0.4845719635486603,
  0.504202663898468,
  0.5017741918563843,
  0.5795130133628845,
  0.5216794610023499]}